# GESTURE CONTROL VOLUME

## Phase 01: Importing libraries and packages.

In [1]:
# install mediaPipe
!pip install mediapipe opencv-python
!pip uninstall mediapipe -y
!pip install mediapipe --user

Defaulting to user installation because normal site-packages is not writeable
Found existing installation: mediapipe 0.10.33
Uninstalling mediapipe-0.10.33:
  Successfully uninstalled mediapipe-0.10.33
  Using cached mediapipe-0.10.33-py3-none-manylinux_2_28_x86_64.whl (11.4 MB)


In [27]:
!pip install pycaw
!pip install comtypes

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [29]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import time
import numpy as np
import uuid
import os
import matplotlib.pyplot as plt
import math

## Phase 02: Initialise mediaPipe to loacate key-points.

In [54]:
import sounddevice as sd
import numpy as np
from scipy.fft import fft, ifft
from scipy.io import wavfile
fs, data = wavfile.read('song.wav')
data = data.astype(np.float32) / 32768.0
if len(data.shape) > 1: data = data[:, 0]
current_audio_frame = 0
bass = 1
pitch = 0
def audio_callback(outdata, frames:1024,time, status):
    global current_audio_frame, data, bass, pitch
    
    # A. Get the next chunk of frames
    start = current_audio_frame
    end = start + frames
    if end > len(data):
        # Loop the song and pad if short
        chunk = data[start:]
        chunk = np.pad(chunk, (0, frames - len(chunk)))
        current_audio_frame = 0
    else:
        chunk = data[start:end]
        current_audio_frame = end
    # B. FFT Processing
    spectrum = fft(chunk)
    # 1. BASS BOOST (First 15 frequency bins)
    # Pinch (0.02) = 4.0x Bass, Open (0.15) = 1.0x Bass
    spectrum[:15] *= float(bass)
    
    # 2. PITCH SHIFT (Bin Shifting)
    # Pinch = Deep (-8 bins), Open = High (+15 bins)
    spectrum = np.roll(spectrum, pitch)

    # C. Reconstruct and Output
    # Ensure the shape is exactly what outdata expects (frames, 1)
    chunk_out = np.real(ifft(spectrum))
    outdata[:] = chunk_out[:frames].reshape(-1, 1)

In [58]:
# initialise camera
cap = cv2.VideoCapture(0, cv2.CAP_V4L2) # V4L2 is best for Zorin Linux
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
BaseOptions = mp.tasks.BaseOptions # tells the system where pre-trained model is located in the system.
HandLandmarker = mp.tasks.vision.HandLandmarker # this is the one that looks for hands.
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions # Thsi holds all the required settings or modifications or things we require. 
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult # defines the structure of result.
VisionRunningMode = mp.tasks.vision.RunningMode # This defines the workFlow mode( input is image, video or live stream ).


current_frame = None
last_vol_time = 0
# Create a hand landmarker instance with the live stream mode:
def print_result(result: HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int): # result stores data from HandLandmarkerResult; output_image stores the live frame; timestamp_ms is the time exactly when the frame has occured and its the value that we need to give after rendering the image.
    # print('hand landmarker result: {}'.format(result))
    # mathematics required for volume control come here
    # detect landmarks
    global current_frame
    global last_vol_time
    global current_audio_frame
    global data, bass, pitch
    # Convert MediaPipe image back to BGR for OpenCV
    frame = cv2.cvtColor(output_image.numpy_view(), cv2.COLOR_RGB2BGR) # converting back to original frame as that is what going to be displayed using openCV.
    h, w, _ = frame.shape # height and width of frame.
    if result.hand_landmarks:
        for landmarks in result.hand_landmarks:
            # generaating skeleton
            
            cv2.line(frame, (int(landmarks[0].x * w), int(landmarks[0].y * h)),(int(landmarks[1].x * w), int(landmarks[1].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[1].x * w), int(landmarks[1].y * h)),(int(landmarks[2].x * w), int(landmarks[2].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[2].x * w), int(landmarks[2].y * h)),(int(landmarks[3].x * w), int(landmarks[3].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[3].x * w), int(landmarks[3].y * h)),(int(landmarks[4].x * w), int(landmarks[4].y * h)),(150, 150, 150) , 3)
            
            cv2.line(frame, (int(landmarks[0].x * w), int(landmarks[0].y * h)),(int(landmarks[5].x * w), int(landmarks[5].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[5].x * w), int(landmarks[5].y * h)),(int(landmarks[6].x * w), int(landmarks[6].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[6].x * w), int(landmarks[6].y * h)),(int(landmarks[7].x * w), int(landmarks[7].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[7].x * w), int(landmarks[7].y * h)),(int(landmarks[8].x * w), int(landmarks[8].y * h)),(150, 150, 150) , 3)
            

            cv2.line(frame, (int(landmarks[0].x * w), int(landmarks[0].y * h)),(int(landmarks[9].x * w), int(landmarks[9].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[9].x * w), int(landmarks[9].y * h)),(int(landmarks[10].x * w), int(landmarks[10].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[10].x * w), int(landmarks[10].y * h)),(int(landmarks[11].x * w), int(landmarks[11].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[11].x * w), int(landmarks[11].y * h)),(int(landmarks[12].x * w), int(landmarks[12].y * h)),(150, 150, 150) , 3)

            cv2.line(frame, (int(landmarks[0].x * w), int(landmarks[0].y * h)),(int(landmarks[13].x * w), int(landmarks[13].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[13].x * w), int(landmarks[13].y * h)),(int(landmarks[14].x * w), int(landmarks[14].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[14].x * w), int(landmarks[14].y * h)),(int(landmarks[15].x * w), int(landmarks[15].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[15].x * w), int(landmarks[15].y * h)),(int(landmarks[16].x * w), int(landmarks[16].y * h)),(150, 150, 150) , 3)

            cv2.line(frame, (int(landmarks[0].x * w), int(landmarks[0].y * h)),(int(landmarks[17].x * w), int(landmarks[17].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[17].x * w), int(landmarks[17].y * h)),(int(landmarks[18].x * w), int(landmarks[18].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[18].x * w), int(landmarks[18].y * h)),(int(landmarks[19].x * w), int(landmarks[19].y * h)),(150, 150, 150) , 3)
            cv2.line(frame, (int(landmarks[19].x * w), int(landmarks[19].y * h)),(int(landmarks[20].x * w), int(landmarks[20].y * h)),(150, 150, 150) , 3)

            for lm in landmarks:
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, (255, 255, 255), -1)
                
            thumb = landmarks[4]
            index = landmarks[8]
            mid_tip = landmarks[12]
            ring_tip = landmarks[16]

            distance_for_loudness = np.sqrt((index.x - thumb.x)**2 + (index.y - thumb.y)**2)
            distance_for_bass = np.sqrt((mid_tip.x - thumb.x)**2 + (mid_tip.y - thumb.y)**2)
            distance_for_pitch = np.sqrt((ring_tip.x - thumb.x)**2 + (ring_tip.y - thumb.y)**2)
            
            # CONVERT TO PIXELS FOR DRAWING
            p1 = (int(thumb.x * w), int(thumb.y * h))
            p2 = (int(index.x * w), int(index.y * h))
            p3 = (int(mid_tip.x * w), int(mid_tip.y * h))
            p4 = (int(ring_tip.x * w), int(ring_tip.y * h))

            # Perform maths to change loudness of volume.
            loudness = int(np.interp(distance_for_loudness, [0.02, 0.25], [0, 100])) # linear interpolation
            bass = int(np.interp(distance_for_bass, [0.02, 0.2], [1, 5]))
            pitch = int(np.interp(distance_for_pitch, [0.02, 0.2], [-10, 20]))
            if time.time() - last_vol_time > 0.1:
                os.system(f"amixer -D pulse sset Master {loudness}% > /dev/null")
                last_vol_time = time.time()

            
            # VISUAL FEEDBACK
            # Line turns Green if pinched, Red if open
            color_for_loudness = (0, 255, 0) if distance_for_loudness < 0.2 else (0, 0, 255)
            color_for_bass = (0, 255, 0) if distance_for_bass < 0.12 else (0, 0, 255)
            color_for_pitch = (0, 255, 0) if distance_for_pitch < 0.12 else (0, 0, 255)
            
            cv2.line(frame, p1, p2, color_for_loudness, 3)
            cv2.circle(frame, p1, 8, (255, 0, 0), cv2.FILLED)
            cv2.circle(frame, p2, 8, (255, 0, 0), cv2.FILLED)

            cv2.line(frame, p1, p3, color_for_bass, 3)
            cv2.circle(frame, p1, 8, (255, 0, 0), cv2.FILLED)
            cv2.circle(frame, p2, 8, (255, 0, 0), cv2.FILLED)

            cv2.line(frame, p1, p4, color_for_pitch, 3)
            cv2.circle(frame, p1, 8, (255, 0, 0), cv2.FILLED)
            cv2.circle(frame, p2, 8, (255, 0, 0), cv2.FILLED)
            
            # Display distance; loudness
            cv2.putText(frame, f"Pinch: {round(distance_for_loudness, 2)}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color_for_loudness, 2)
            cv2.putText(frame, f"VOL: {loudness}%", (50, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, color_for_loudness, 2)
            cv2.putText(frame, f"BASS: {bass}", (50, 350), cv2.FONT_HERSHEY_SIMPLEX, 1, color_for_bass, 2)
            cv2.putText(frame, f"PITCH: {pitch}", (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 1, color_for_pitch, 2)
            
    current_frame = frame
    
        
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=print_result)


with HandLandmarker.create_from_options(options) as landmarker:
  # The landmarker is initialized. Use it here.
    stream = sd.OutputStream(samplerate=fs, channels=1, callback=audio_callback)
    with stream:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Ignoring empty camera frame.")
                continue
            # image processing to send frame to AI
            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # MediaPipe needs RGB, but OpenCV gives BGR; therefore this conversion is used.
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame) # setting up image format and data.
            frame_timestamp_ms = int(time.time() * 1000) # setting time to render single frame per second
            landmarker.detect_async(mp_image, frame_timestamp_ms) # send time_stamp and modified frame to landmarker.
            
           # displayinf the frame
            if current_frame is not None:
                cv2.imshow('Zorin Hand Tracking', current_frame)
            else:
                # If AI hasn't processed yet, just show the raw frame
                cv2.imshow('Zorin Hand Tracking', frame)
    
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1774045071.971120   21528 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774045071.972304   21547 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: REMBRANDT (rembrandt, LLVM 15.0.7, DRM 3.57, 6.8.0-106-generic)
W0000 00:00:1774045071.977786   21538 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774045071.983828   21541 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Dealing with bass, pitch and real time audio playback system

In [36]:
!pip install scipy sounddevice

Defaulting to user installation because normal site-packages is not writeable
